# frc-repack — copy ONLY the model artifacts out of the frc-train output

The training kernel's output is multi-GB (it includes the 2,400 auto-labeled frames). This
kernel attaches that output and copies just the deployable bits to a clean /kaggle/working
so the pull is ~60MB instead of ~6GB. No GPU, no internet — pure file copy.

In [ ]:
import shutil, glob, os
os.makedirs('/kaggle/working/weights', exist_ok=True)
# OpenVINO model dirs (the deployable, CUDA-free detectors)
for s in glob.glob('/kaggle/input/**/ball_openvino_model', recursive=True) + glob.glob('/kaggle/input/**/robot_openvino_model', recursive=True):
    name = os.path.basename(s)
    shutil.copytree(s, f'/kaggle/working/{name}', dirs_exist_ok=True)
    print('copied dir', name)
# packaged zip
for z in glob.glob('/kaggle/input/**/frc_model_openvino.zip', recursive=True):
    shutil.copy(z, '/kaggle/working/'); print('copied zip', os.path.basename(z))
# best.pt weights (PyTorch) for fallback / the teacher-vs-student render
for p in glob.glob('/kaggle/input/**/runs/*/weights/best.pt', recursive=True):
    tag = p.split('/runs/')[1].split('/')[0]
    shutil.copy(p, f'/kaggle/working/weights/{tag}_best.pt'); print('copied', tag, 'best.pt')
# results.csv (per-epoch metrics) for both students, if present
for c in glob.glob('/kaggle/input/**/runs/*/results.csv', recursive=True):
    tag = c.split('/runs/')[1].split('/')[0]
    shutil.copy(c, f'/kaggle/working/weights/{tag}_results.csv'); print('copied', tag, 'results.csv')
print('OUTPUT:', sorted(os.listdir('/kaggle/working')))